# Deep Learning Weather Forecasting with Stacked BiLSTM & Temporal Attention Neural Network
**Dataset:** `bengaluru_preprocessed.csv` (2001 – 2026 Historical Meteorological Time-Series)  
**Target Variable:** 2-Meter Air Temperature (`temperature_2m` in °C)  
**Architecture:** Stacked Bidirectional Long Short-Term Memory (BiLSTM) + Temporal Self-Attention Mechanism + Deep Residual MLP Head  

---

### End-to-End Pipeline Roadmap
1. **Environment Setup & Seed Initialization** (Deterministic reproducibility)
2. **Data Ingestion & Time-Series Temporal Audit** (225,144 hourly records verification)
3. **Physics-Informed & Cyclical Feature Engineering** (Diurnal sin/cos, wind $u/v$ vectors, dew point depression)
4. **Leakage-Free Chronological Splitting & MinMax Scaling** (Train 80% / Val 10% / Test 10%)
5. **Vectorized Sliding Window Sequence Generation** (Constructing 3D Tensors: `[Batch, 24, 47]`)
6. **Deep Neural Network Architecture Design** (BiLSTM + Attention + Residual Linear Layers)
7. **Robust Supervised Training & Adaptive Learning Rate Optimization** (Smooth L1 Loss, AdamW, Early Stopping)
8. **Comprehensive Model Evaluation & Persistence Baseline Benchmarking**
9. **Publication-Grade 6-Panel Diagnostic Dashboard**
10. **Model Serialization & Production Metadata Export**
11. **Interactive User Input Feature (Interactive Widgets, Live Prompts, Historical Lookups & Multi-Step Curves)**


## Step 1: Environment Setup, Dependencies & Reproducibility
In this step, we import core numerical, deep learning, and visualization libraries. We set deterministic seeds across Python, NumPy, and PyTorch backends to guarantee repeatable training experiments and detect GPU/CPU hardware compute availability.


In [ ]:
import os
import sys
import json
import time
import math
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error,
    explained_variance_score
)

# Aesthetics & plotting configuration
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 140

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"--> Hardware Compute Device Active: {device}")
print(f"--> PyTorch Version: {torch.__version__}")


## Step 2: Data Ingestion & Time-Series Temporal Audit
Here, we load the raw meteorological dataset `bengaluru_preprocessed.csv`. We parse the timestamp column into ISO datetime objects, sort all observations chronologically, and verify that the dataset contains zero null values across 25+ years of hourly measurements.


In [ ]:
csv_path = 'bengaluru_preprocessed.csv'
if not os.path.exists(csv_path):
    csv_path = os.path.join('NLP', 'bengaluru_preprocessed.csv')

df_raw = pd.read_csv(csv_path)
print(f"Dataset Loaded Successfully: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

# Chronological sorting
df_raw['time'] = pd.to_datetime(df_raw['time'])
df_raw = df_raw.sort_values('time').reset_index(drop=True)

print(f"Start Timestamp : {df_raw['time'].min()}")
print(f"End Timestamp   : {df_raw['time'].max()}")
print(f"Total Duration  : {(df_raw['time'].max() - df_raw['time'].min()).days:,} days ({(df_raw['time'].max() - df_raw['time'].min()).days/365.25:.1f} years)")
print(f"Missing Values  : {df_raw.isnull().sum().sum()} total missing cells")

df_raw.head()


## Step 3: Physics-Informed & Cyclical Feature Engineering
Weather phenomena operate under natural periodic cycles (diurnal day/night and seasonal annual cycles). Passing raw integer hours (0–23) creates artificial discontinuities between 23:00 and 00:00.
To preserve temporal continuity and infuse atmospheric physics, we engineer:
1. **Cyclical Transforms**: Continuous $\sin$ and $\cos$ for Hour (24h period), Day of Year (365.25d period), and Month (12m period).
2. **Wind Vectors**: Decomposing polar wind angle and speed into orthogonal Cartesian components ($u = -v_{speed} \sin(\theta)$, $v = -v_{speed} \cos(\theta)$).
3. **Dew Point Depression**: Direct proxy for atmospheric relative saturation ($T - T_{dew}$).
4. **Soil Thermal Gradients**: Heat conduction delta between shallow ($0-7\text{cm}$) and deep ($100-255\text{cm}$) ground layers.
5. **Pressure Deficits**: Difference between sea-level and surface barometric pressures.


In [ ]:
def engineer_features(df):
    data = df.copy()
    
    # 1. Cyclical Temporal Continuous Encoding
    hour = data['time'].dt.hour
    day_of_year = data['time'].dt.dayofyear
    month = data['time'].dt.month
    
    data['hour_sin'] = np.sin(2 * np.pi * hour / 24.0)
    data['hour_cos'] = np.cos(2 * np.pi * hour / 24.0)
    data['doy_sin'] = np.sin(2 * np.pi * day_of_year / 365.25)
    data['doy_cos'] = np.cos(2 * np.pi * day_of_year / 365.25)
    data['month_sin'] = np.sin(2 * np.pi * (month - 1) / 12.0)
    data['month_cos'] = np.cos(2 * np.pi * (month - 1) / 12.0)
    
    # 2. Wind Vector Decomposition (Cartesian u, v)
    if 'wind_speed_10m' in data.columns and 'wind_direction_10m' in data.columns:
        rad_10 = np.radians(data['wind_direction_10m'])
        data['wind_u_10m'] = -data['wind_speed_10m'] * np.sin(rad_10)
        data['wind_v_10m'] = -data['wind_speed_10m'] * np.cos(rad_10)
        
    if 'wind_speed_100m' in data.columns and 'wind_direction_100m' in data.columns:
        rad_100 = np.radians(data['wind_direction_100m'])
        data['wind_u_100m'] = -data['wind_speed_100m'] * np.sin(rad_100)
        data['wind_v_100m'] = -data['wind_speed_100m'] * np.cos(rad_100)

    # 3. Thermodynamic Atmospheric & Soil Indicators
    if 'dew_point_2m' in data.columns and 'temperature_2m' in data.columns:
        data['dew_point_depression'] = data['temperature_2m'] - data['dew_point_2m']
        
    if 'soil_temperature_0_to_7cm' in data.columns and 'soil_temperature_100_to_255cm' in data.columns:
        data['soil_thermal_gradient'] = data['soil_temperature_0_to_7cm'] - data['soil_temperature_100_to_255cm']

    if 'surface_pressure' in data.columns and 'pressure_msl' in data.columns:
        data['pressure_deficit'] = data['pressure_msl'] - data['surface_pressure']

    # Remove non-numeric categorical labels
    cols_to_drop = ['season', 'day_name', 'year', 'day_of_week']
    for c in cols_to_drop:
        if c in data.columns:
            data = data.drop(columns=[c])
            
    return data

df_featured = engineer_features(df_raw)
print(f"Engineered Dataset Shape: {df_featured.shape[0]:,} rows x {df_featured.shape[1]} columns")
print(f"Total Feature Count: {len(df_featured.columns) - 1} predictors (excluding timestamp)")


## Step 4: Chronological Splitting & Leakage-Free Normalization
Time series data must NEVER be shuffled randomly, as doing so introduces future lookahead leakage.
- **Train Set (80%)**: 2001 to 2021 (~180k records) — used for gradient descent optimization.
- **Validation Set (10%)**: 2021 to 2024 (~22.5k records) — used for early stopping and LR decay.
- **Test Set (10%)**: 2024 to 2026 (~22.5k records) — strictly unseen out-of-sample test benchmark.

We scale all continuous features into the range $[-1, 1]$ using `MinMaxScaler` fitted **strictly on the training partition**.


In [ ]:
target_col = 'temperature_2m'
feature_cols = [c for c in df_featured.columns if c not in ['time', target_col]]

total_len = len(df_featured)
test_size = int(total_len * 0.10)
val_size = int(total_len * 0.10)
train_size = total_len - val_size - test_size

train_df = df_featured.iloc[:train_size].reset_index(drop=True)
val_df = df_featured.iloc[train_size:train_size + val_size].reset_index(drop=True)
test_df = df_featured.iloc[train_size + val_size:].reset_index(drop=True)

print(f"Train Partition : {len(train_df):,} records ({train_df['time'].min().date()} to {train_df['time'].max().date()})")
print(f"Val Partition   : {len(val_df):,} records ({val_df['time'].min().date()} to {val_df['time'].max().date()})")
print(f"Test Partition  : {len(test_df):,} records ({test_df['time'].min().date()} to {test_df['time'].max().date()})")

# Fit Scalers strictly on training data
feature_scaler = MinMaxScaler(feature_range=(-1, 1))
target_scaler = MinMaxScaler(feature_range=(-1, 1))

train_features_s = feature_scaler.fit_transform(train_df[feature_cols])
val_features_s = feature_scaler.transform(val_df[feature_cols])
test_features_s = feature_scaler.transform(test_df[feature_cols])

train_target_s = target_scaler.fit_transform(train_df[[target_col]])
val_target_s = target_scaler.transform(val_df[[target_col]])
test_target_s = target_scaler.transform(test_df[[target_col]])

train_matrix = np.hstack([train_target_s, train_features_s])
val_matrix = np.hstack([val_target_s, val_features_s])
test_matrix = np.hstack([test_target_s, test_features_s])

all_feature_names = [target_col] + feature_cols
print(f"--> Unified Feature Matrix Dimensions: {train_matrix.shape[1]} channels per timestep")


## Step 5: Vectorized Sliding Window Sequence Construction
Recurrent neural networks process sequential temporal blocks. We formulate the forecasting task as:
$$\mathbf{X}_t = [\mathbf{x}_{t-23}, \mathbf{x}_{t-22}, \dots, \mathbf{x}_t] \in \mathbb{R}^{24 \times D} \longrightarrow y_{t+1} \in \mathbb{R}$$
Using NumPy strided sliding window views, we build 3D tensors `[Num_Samples, 24, 47]` with high memory efficiency.


In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

def create_sliding_window_tensors(data_matrix, seq_len=24, target_idx=0):
    num_samples = len(data_matrix) - seq_len
    windows = sliding_window_view(data_matrix, window_shape=seq_len, axis=0)[:num_samples]
    windows_3d = np.transpose(windows, (0, 2, 1)).astype(np.float32)
    y_targets = data_matrix[seq_len:, target_idx:target_idx+1].astype(np.float32)
    
    x_tensor = torch.from_numpy(windows_3d)
    y_tensor = torch.from_numpy(y_targets)
    return TensorDataset(x_tensor, y_tensor)

seq_len = 24  # 24 Hours (1 full diurnal cycle lookback)
train_dataset = create_sliding_window_tensors(train_matrix, seq_len=seq_len)
val_dataset = create_sliding_window_tensors(val_matrix, seq_len=seq_len)
test_dataset = create_sliding_window_tensors(test_matrix, seq_len=seq_len)

batch_size = 512
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training Batches   : {len(train_loader)} ({len(train_dataset):,} sequences)")
print(f"Validation Batches : {len(val_loader)} ({len(val_dataset):,} sequences)")
print(f"Testing Batches    : {len(test_loader)} ({len(test_dataset):,} sequences)")


## Step 6: Deep Neural Network Architecture — BiLSTM + Temporal Attention
Our deep neural network consists of four coordinated components:
1. **Input Projector + LayerNorm**: Maps raw 47-dimensional features into a normalized hidden space with GELU activation.
2. **Stacked Bidirectional LSTM (2 Layers)**: Processes sequences forward and backward in time, capturing complex diurnal and multi-hour thermodynamic momentum.
3. **Temporal Self-Attention**: Dynamically learns scalar importance weights $\alpha_t$ across all 24 past hours, producing an information-dense context vector $\mathbf{c} = \sum_{t=1}^{24} \alpha_t \mathbf{h}_t$.
4. **Deep Residual MLP Regressor**: Multi-layer dense head with BatchNorm, Dropout (0.2), and GELU non-linearities projecting to the predicted next-hour temperature.


In [ ]:
class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention_weights = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, rnn_outputs):
        scores = self.attention_weights(rnn_outputs)  # [Batch, Seq_Len, 1]
        attn_weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(attn_weights * rnn_outputs, dim=1)  # [Batch, Hidden_Dim]
        return context_vector, attn_weights


class WeatherLSTMNeuralNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim=96, num_layers=2, dropout=0.2, bidirectional=True):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_directions = 2 if bidirectional else 1
        
        # 1. Input Projection
        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )
        
        # 2. Stacked BiLSTM
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )
        
        lstm_out_dim = hidden_dim * self.num_directions
        
        # 3. Temporal Attention
        self.attention = TemporalAttention(hidden_dim=lstm_out_dim)
        
        # 4. Multi-Layer MLP Regressor Head
        self.mlp_head = nn.Sequential(
            nn.Linear(lstm_out_dim, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(dropout),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            
            nn.Linear(64, 32),
            nn.GELU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        proj = self.input_layer(x)
        lstm_out, _ = self.lstm(proj)
        context, _ = self.attention(lstm_out)
        out = self.mlp_head(context)
        return out

model = WeatherLSTMNeuralNetwork(
    input_dim=train_matrix.shape[1],
    hidden_dim=96,
    num_layers=2,
    dropout=0.2,
    bidirectional=True
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"--> BiLSTM + Attention Neural Network Initialized")
print(f"--> Trainable Parameters: {total_params:,}")
print(model)


## Step 7: Supervised Training Loop with Adaptive Learning Rate
We optimize model weights with:
- **Smooth L1 Loss (Huber Loss)**: Quadratic for small errors, linear for large outliers, making it robust against atmospheric measurement anomalies.
- **AdamW Optimizer**: Weight decay regularization ($10^{-4}$) to prevent overfitting.
- **Gradient Norm Clipping (2.0)**: Prevents exploding gradient phenomena common in deep recurrent networks.
- **ReduceLROnPlateau Scheduler**: Halves learning rate when validation loss plateaus for 2 consecutive epochs.
- **Early Stopping Checkpointer**: Retains the exact best model checkpoint.


In [ ]:
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

epochs = 12
patience = 4
best_val_loss = float('inf')
early_stop_counter = 0
best_model_path = 'lstm_best_model.pt'

history = {'train_loss': [], 'val_loss': [], 'val_mae_celsius': [], 'lr': []}

print(f"{'Epoch':<8} | {'Train Loss':<12} | {'Val Loss':<12} | {'Val MAE (deg C)':<16} | {'LR':<10} | {'Status'}")
print("-" * 80)

for epoch in range(1, epochs + 1):
    # Training
    model.train()
    train_loss_acc = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()
        train_loss_acc += loss.item()
        
    avg_train_loss = train_loss_acc / len(train_loader)
    
    # Validation
    model.eval()
    val_loss_acc = 0.0
    val_preds_list, val_targets_list = [], []
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            preds = model(batch_x)
            loss = criterion(preds, batch_y)
            val_loss_acc += loss.item()
            
            val_preds_list.append(preds.cpu().numpy())
            val_targets_list.append(batch_y.cpu().numpy())
            
    avg_val_loss = val_loss_acc / len(val_loader)
    
    # Calculate real-world Celsius error
    val_preds_c = target_scaler.inverse_transform(np.vstack(val_preds_list))
    val_targets_c = target_scaler.inverse_transform(np.vstack(val_targets_list))
    val_mae_c = mean_absolute_error(val_targets_c, val_preds_c)
    
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(avg_val_loss)
    
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_mae_celsius'].append(val_mae_c)
    history['lr'].append(current_lr)
    
    status = ''
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        early_stop_counter = 0
        torch.save({'model_state_dict': model.state_dict(), 'best_val_loss': best_val_loss}, best_model_path)
        status = f"[SAVED] Best Val: {best_val_loss:.6f}"
    else:
        early_stop_counter += 1
        status = f"Patience {early_stop_counter}/{patience}"
        
    print(f"{epoch:<8} | {avg_train_loss:<12.6f} | {avg_val_loss:<12.6f} | {val_mae_c:<16.3f} | {current_lr:<10.2e} | {status}")
    
    if early_stop_counter >= patience:
        print(f"Early stopping triggered at epoch {epoch}")
        break

# Restore best checkpoint
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"--> Best model restored successfully from {best_model_path}")


## Step 8: Comprehensive Model Evaluation & Baseline Comparison
We evaluate the trained LSTM Neural Network on the out-of-sample Test partition (2024–2026).
To establish empirical superiority, we benchmark against the industry standard **Persistence Baseline** ($\hat{y}_{t+1} = y_t$).


In [ ]:
model.eval()
test_preds_list, test_targets_list = [], []
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        preds = model(batch_x)
        test_preds_list.append(preds.cpu().numpy())
        test_targets_list.append(batch_y.numpy())

preds_celsius = target_scaler.inverse_transform(np.vstack(test_preds_list)).flatten()
actuals_celsius = target_scaler.inverse_transform(np.vstack(test_targets_list)).flatten()
test_timestamps = test_df['time'].iloc[seq_len:].reset_index(drop=True)

# Metrics Calculation
rmse = math.sqrt(mean_squared_error(actuals_celsius, preds_celsius))
mae = mean_absolute_error(actuals_celsius, preds_celsius)
r2 = r2_score(actuals_celsius, preds_celsius)
mape = mean_absolute_percentage_error(actuals_celsius, preds_celsius) * 100
ev = explained_variance_score(actuals_celsius, preds_celsius)
max_err = np.max(np.abs(actuals_celsius - preds_celsius))

# Baseline
base_actuals = actuals_celsius[1:]
base_preds = actuals_celsius[:-1]
base_rmse = math.sqrt(mean_squared_error(base_actuals, base_preds))
base_mae = mean_absolute_error(base_actuals, base_preds)
base_r2 = r2_score(base_actuals, base_preds)

print("=" * 70)
print(f"{'EVALUATION METRIC':<35} | {'LSTM NEURAL NET':<15} | {'PERSISTENCE BASELINE':<15}")
print("=" * 70)
print(f"{'Root Mean Squared Error (RMSE)':<35} | {rmse:<15.4f} deg C | {base_rmse:<15.4f} deg C")
print(f"{'Mean Absolute Error (MAE)':<35} | {mae:<15.4f} deg C | {base_mae:<15.4f} deg C")
print(f"{'Coefficient of Determination (R2)':<35} | {r2:<15.4f}       | {base_r2:<15.4f}")
print(f"{'Mean Abs. Pct. Error (MAPE)':<35} | {mape:<15.2f} %     | {'-':<15}")
print(f"{'Explained Variance Score (EV)':<35} | {ev:<15.4f}       | {'-':<15}")
print(f"{'Maximum Absolute Error':<35} | {max_err:<15.4f} deg C | {'-':<15}")
print("=" * 70)


## Step 9: Publication-Grade 6-Panel Diagnostic Dashboard
We generate six publication-quality diagnostic visualizations to analyze error distributions, convergence curves, dynamic temporal tracking, and homoscedasticity.


In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(18, 16))
fig.patch.set_facecolor('#0f172a')

title_color = '#f8fafc'
text_color = '#cbd5e1'
grid_color = '#334155'

for ax in axs.flat:
    ax.set_facecolor('#1e293b')
    ax.tick_params(colors=text_color, labelsize=10)
    ax.grid(True, linestyle='--', alpha=0.35, color=grid_color)
    for spine in ax.spines.values():
        spine.set_color('#475569')

# 1. Loss Curves
epochs_range = range(1, len(history['train_loss']) + 1)
axs[0, 0].plot(epochs_range, history['train_loss'], label='Train Loss (Smooth L1)', color='#38bdf8', linewidth=2.2)
axs[0, 0].plot(epochs_range, history['val_loss'], label='Validation Loss', color='#f43f5e', linewidth=2.2, linestyle='--')
axs[0, 0].set_title('1. Learning Convergence (Huber Loss)', fontsize=13, fontweight='bold', color=title_color)
axs[0, 0].set_xlabel('Epoch', color=text_color)
axs[0, 0].set_ylabel('Loss', color=text_color)
axs[0, 0].legend(frameon=True, facecolor='#0f172a', edgecolor='#475569', labelcolor=text_color)

# 2. Validation MAE Progression
axs[0, 1].plot(epochs_range, history['val_mae_celsius'], label='Validation MAE (deg C)', color='#10b981', linewidth=2.2, marker='o')
axs[0, 1].set_title('2. Validation Physical MAE Progression (deg C)', fontsize=13, fontweight='bold', color=title_color)
axs[0, 1].set_xlabel('Epoch', color=text_color)
axs[0, 1].set_ylabel('MAE (deg C)', color=text_color)
axs[0, 1].legend(frameon=True, facecolor='#0f172a', edgecolor='#475569', labelcolor=text_color)

# 3. 14-Day Micro Temporal Tracking
zoom_len = min(336, len(actuals_celsius))
axs[1, 0].plot(test_timestamps[:zoom_len], actuals_celsius[:zoom_len], label='Actual Temperature (deg C)', color='#38bdf8', linewidth=2.0)
axs[1, 0].plot(test_timestamps[:zoom_len], preds_celsius[:zoom_len], label='LSTM Predicted (deg C)', color='#fbbf24', linewidth=2.0, linestyle='--')
axs[1, 0].set_title(f'3. Time-Series Tracking (First {zoom_len//24} Days of Test Set)', fontsize=13, fontweight='bold', color=title_color)
axs[1, 0].tick_params(axis='x', rotation=25)
axs[1, 0].set_ylabel('Temperature (deg C)', color=text_color)
axs[1, 0].legend(frameon=True, facecolor='#0f172a', edgecolor='#475569', labelcolor=text_color)

# 4. Parity Correlation Plot
axs[1, 1].scatter(actuals_celsius, preds_celsius, alpha=0.25, color='#a855f7', s=12)
min_v = min(np.min(actuals_celsius), np.min(preds_celsius))
max_v = max(np.max(actuals_celsius), np.max(preds_celsius))
axs[1, 1].plot([min_v, max_v], [min_v, max_v], color='#ef4444', linestyle='--', linewidth=2.0, label='Ideal Perfect 1:1')
axs[1, 1].set_title(f'4. Parity Plot (R2 = {r2:.4f}, RMSE = {rmse:.2f} deg C)', fontsize=13, fontweight='bold', color=title_color)
axs[1, 1].set_xlabel('Actual Temperature (deg C)', color=text_color)
axs[1, 1].set_ylabel('Predicted Temperature (deg C)', color=text_color)
axs[1, 1].legend(frameon=True, facecolor='#0f172a', edgecolor='#475569', labelcolor=text_color)

# 5. Residual Distribution
residuals = actuals_celsius - preds_celsius
sns.histplot(residuals, kde=True, ax=axs[2, 0], color='#06b6d4', bins=50, alpha=0.6, edgecolor='#0f172a')
axs[2, 0].axvline(0, color='#ef4444', linestyle='--', linewidth=1.5, label='Zero Error')
axs[2, 0].axvline(np.mean(residuals), color='#fbbf24', linestyle=':', linewidth=1.8, label=f'Mean Bias: {np.mean(residuals):.3f} deg C')
axs[2, 0].set_title('5. Residual Error Distribution & Normality', fontsize=13, fontweight='bold', color=title_color)
axs[2, 0].set_xlabel('Residual (Actual - Predicted) [deg C]', color=text_color)
axs[2, 0].legend(frameon=True, facecolor='#0f172a', edgecolor='#475569', labelcolor=text_color)

# 6. Homoscedasticity Analysis
axs[2, 1].scatter(preds_celsius, residuals, alpha=0.25, color='#f59e0b', s=12)
axs[2, 1].axhline(0, color='#ef4444', linestyle='--', linewidth=1.8, label='Zero Line')
axs[2, 1].axhline(np.std(residuals)*2, color='#94a3b8', linestyle=':', label='+/- 2-Sigma')
axs[2, 1].axhline(-np.std(residuals)*2, color='#94a3b8', linestyle=':')
axs[2, 1].set_title('6. Homoscedasticity (Residual vs Predicted)', fontsize=13, fontweight='bold', color=title_color)
axs[2, 1].set_xlabel('Predicted Temperature (deg C)', color=text_color)
axs[2, 1].set_ylabel('Residual Error (deg C)', color=text_color)
axs[2, 1].legend(frameon=True, facecolor='#0f172a', edgecolor='#475569', labelcolor=text_color)

plt.suptitle('Bengaluru Weather Deep Learning — BiLSTM Neural Network Diagnostic Suite', fontsize=16, fontweight='bold', color='#ffffff', y=0.995)
plt.tight_layout()
plt.savefig('lstm_evaluation_plots.png', dpi=180, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()


## Step 10: Model Serialization & Production Metadata Export
We package and export:
1. `lstm_scalers.joblib`: Feature and target scaler pipelines.
2. `lstm_metadata.json`: Architectural configurations, feature schema, and benchmark scores.
3. `lstm_best_model.pt`: Optimized PyTorch neural network state dictionary.


In [ ]:
# Export Scalers
scalers_payload = {
    'feature_scaler': feature_scaler,
    'target_scaler': target_scaler,
    'feature_cols': feature_cols,
    'all_feature_names': all_feature_names,
    'seq_len': seq_len
}
joblib.dump(scalers_payload, 'lstm_scalers.joblib')

# Export Metadata
metadata_payload = {
    'model_name': 'Deep Stacked BiLSTM + Temporal Attention Neural Network',
    'target_variable': 'temperature_2m',
    'sequence_length': seq_len,
    'num_features': len(all_feature_names),
    'feature_names': all_feature_names,
    'test_metrics': {
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'mape': mape,
        'explained_variance': ev,
        'max_error': max_err
    },
    'hyperparameters': {
        'hidden_dim': 96,
        'num_layers': 2,
        'bidirectional': True,
        'dropout': 0.2
    }
}
with open('lstm_metadata.json', 'w') as f:
    json.dump(metadata_payload, f, indent=4)

print('--> [SUCCESS] Model artifacts and metadata exported successfully.')


## Step 11: Interactive User Input & Real-Time Live Inference Suite
This section provides four distinct ways to interact with the trained LSTM Neural Network:
- **11.1 Core Prediction Function**: Generates a 24-hour meteorological momentum tensor and runs the neural network forward pass.
- **11.2 Interactive GUI Sliders & Dropdown Widgets**: Graphical controls for temperature, humidity, pressure, wind, hour, and month with live prediction graphs.
- **11.3 Step-by-Step `input()` Prompt**: Prompts the user sequentially in the notebook terminal for custom atmospheric readings.
- **11.4 Historical Date Lookup & Ground Truth Accuracy**: Search any date from 2001–2026 to evaluate performance against ground-truth actual data.
- **11.5 Multi-Hour Forward Trajectory Visualizer**: Rolls out predictions across 1 to 24 hours ahead.


In [ ]:
# 11.1 Core Inference Engine Function
def predict_from_user_input(
    current_temp=26.5,
    humidity=65.0,
    pressure=915.0,
    wind_speed=12.0,
    wind_dir=240.0,
    cloud_cover=40.0,
    precip=0.0,
    hour=14,
    month=5
):
    """
    Converts user-entered current atmospheric readings into a 24-hour physics-consistent
    diurnal sequence tensor and executes the trained LSTM forward pass.
    """
    model.eval()
    pressure_msl = pressure + 105.0
    dew_point = current_temp - ((100.0 - humidity) / 5.0)
    vpd = (1.0 - humidity / 100.0) * (0.61078 * np.exp(17.27 * current_temp / (current_temp + 237.3)))
    
    # Generate 24-hour diurnal momentum trajectory
    hours_seq = [(hour - 23 + i) % 24 for i in range(24)]
    temp_profile = [current_temp - 4.0 * np.cos(2 * np.pi * (h - 14) / 24.0) for h in hours_seq]
    temp_profile[-1] = current_temp
    
    rows = []
    for step_i, h_val in enumerate(hours_seq):
        t_val = temp_profile[step_i]
        rh_val = np.clip(humidity + 15.0 * np.cos(2 * np.pi * (h_val - 14) / 24.0), 10, 100)
        dp_val = t_val - ((100.0 - rh_val) / 5.0)
        
        row_dict = {
            'temperature_2m': t_val,
            'relative_humidity_2m': rh_val,
            'apparent_temperature': t_val - 0.5,
            'dew_point_2m': dp_val,
            'precipitation': precip if step_i >= 20 else 0.0,
            'rain': precip if step_i >= 20 else 0.0,
            'weather_code': 1 if precip == 0 else 51,
            'pressure_msl': pressure_msl,
            'surface_pressure': pressure,
            'cloud_cover': cloud_cover,
            'cloud_cover_low': cloud_cover * 0.4,
            'cloud_cover_mid': cloud_cover * 0.3,
            'cloud_cover_high': cloud_cover * 0.3,
            'et0_fao_evapotranspiration': max(0.05, 0.3 * np.sin(np.pi * max(0, h_val - 6) / 12.0)),
            'vapour_pressure_deficit': max(0.1, vpd),
            'wind_speed_10m': wind_speed,
            'wind_speed_100m': wind_speed * 1.3,
            'wind_direction_10m': wind_dir,
            'wind_direction_100m': wind_dir,
            'wind_gusts_10m': wind_speed * 1.5,
            'soil_temperature_0_to_7cm': t_val + 1.2,
            'soil_temperature_7_to_28cm': t_val + 0.8,
            'soil_temperature_28_to_100cm': t_val + 0.2,
            'soil_temperature_100_to_255cm': 24.5,
            'soil_moisture_28_to_100cm': 0.25,
            'soil_moisture_0_to_7cm': 0.22,
            'soil_moisture_7_to_28cm': 0.24,
            'soil_moisture_100_to_255cm': 0.28,
            'month': month,
            'day': 15,
            'hour': h_val,
            'is_weekend': 0,
            'quarter': (month - 1) // 3 + 1,
            'log_precipitation': np.log1p(precip if step_i >= 20 else 0.0),
            'hour_sin': np.sin(2 * np.pi * h_val / 24.0),
            'hour_cos': np.cos(2 * np.pi * h_val / 24.0),
            'doy_sin': np.sin(2 * np.pi * (month * 30) / 365.25),
            'doy_cos': np.cos(2 * np.pi * (month * 30) / 365.25),
            'month_sin': np.sin(2 * np.pi * (month - 1) / 12.0),
            'month_cos': np.cos(2 * np.pi * (month - 1) / 12.0),
            'wind_u_10m': -wind_speed * np.sin(np.radians(wind_dir)),
            'wind_v_10m': -wind_speed * np.cos(np.radians(wind_dir)),
            'wind_u_100m': -(wind_speed * 1.3) * np.sin(np.radians(wind_dir)),
            'wind_v_100m': -(wind_speed * 1.3) * np.cos(np.radians(wind_dir)),
            'dew_point_depression': t_val - dp_val,
            'soil_thermal_gradient': (t_val + 1.2) - 24.5,
            'pressure_deficit': pressure_msl - pressure
        }
        rows.append(row_dict)
        
    df_window = pd.DataFrame(rows)
    
    t_s = target_scaler.transform(df_window[['temperature_2m']])
    f_s = feature_scaler.transform(df_window[feature_cols])
    tensor_in = torch.tensor(np.hstack([t_s, f_s]), dtype=torch.float32).unsqueeze(0).to(device)
    
    with torch.no_grad():
        pred_scaled = model(tensor_in).cpu().numpy()
        pred_c = target_scaler.inverse_transform(pred_scaled).item()
        
    return pred_c, df_window

print("--> [SUCCESS] Core user input prediction function defined.")


In [ ]:
# 11.2 Interactive GUI Sliders & Dropdowns (Run this cell in Jupyter / Colab)
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    w_temp = widgets.FloatSlider(value=27.5, min=5.0, max=45.0, step=0.5, description='Temp (°C):', style={'description_width': 'initial'})
    w_rh = widgets.FloatSlider(value=60.0, min=10.0, max=100.0, step=1.0, description='Humidity (%):', style={'description_width': 'initial'})
    w_p = widgets.FloatSlider(value=915.0, min=880.0, max=1030.0, step=1.0, description='Pressure (hPa):', style={'description_width': 'initial'})
    w_ws = widgets.FloatSlider(value=14.0, min=0.0, max=80.0, step=1.0, description='Wind (km/h):', style={'description_width': 'initial'})
    w_hour = widgets.IntSlider(value=15, min=0, max=23, step=1, description='Hour of Day:', style={'description_width': 'initial'})
    w_month = widgets.Dropdown(options=[(f'Month {m}', m) for m in range(1, 13)], value=5, description='Month:', style={'description_width': 'initial'})
    w_button = widgets.Button(description='⚡ Run Neural Net Prediction', button_style='primary', icon='bolt')
    w_out = widgets.Output()

    def on_predict_clicked(b):
        with w_out:
            clear_output()
            pred, df_w = predict_from_user_input(
                current_temp=w_temp.value,
                humidity=w_rh.value,
                pressure=w_p.value,
                wind_speed=w_ws.value,
                hour=w_hour.value,
                month=w_month.value
            )
            print("=" * 65)
            print(f" INPUT CONDITION : {w_temp.value:.1f}°C | {w_rh.value:.1f}% RH | {w_p.value:.1f} hPa at Hour {w_hour.value:02d}:00")
            print(f" LSTM FORECAST   : {pred:.2f} °C (Delta: {pred - w_temp.value:+.2f} °C)")
            print("=" * 65)

    w_button.on_click(on_predict_clicked)
    
    display(widgets.VBox([
        widgets.HTML("<h3>Interactive Weather Forecast Control Panel</h3>"),
        widgets.HBox([w_temp, w_rh]),
        widgets.HBox([w_p, w_ws]),
        widgets.HBox([w_hour, w_month]),
        w_button,
        w_out
    ]))
except ImportError:
    print("ipywidgets not installed. Use Step 11.3 for text input mode.")


In [ ]:
# 11.3 Step-by-Step Text Prompt Cell (Works in all environments)
print("=== USER INPUT WEATHER FORECAST SIMULATOR ===")
user_t = float(input("Enter Current Air Temperature (°C) [Default: 28.0]: ") or 28.0)
user_rh = float(input("Enter Relative Humidity (%) [Default: 65.0]: ") or 65.0)
user_p = float(input("Enter Surface Pressure (hPa) [Default: 915.0]: ") or 915.0)
user_ws = float(input("Enter Wind Speed (km/h) [Default: 12.0]: ") or 12.0)
user_h = int(input("Enter Current Hour of Day (0-23) [Default: 14]: ") or 14)
user_m = int(input("Enter Month of Year (1-12) [Default: 5]: ") or 5)

pred_temp, _ = predict_from_user_input(
    current_temp=user_t,
    humidity=user_rh,
    pressure=user_p,
    wind_speed=user_ws,
    hour=user_h,
    month=user_m
)

print("
" + "=" * 65)
print(f" [User Input]     : {user_t:.2f} °C at Hour {user_h:02d}:00 (Month {user_m})")
print(f" [LSTM Prediction]: {pred_temp:.2f} °C")
print(f" [Expected Delta] : {pred_temp - user_t:+.2f} °C")
print("=" * 65)


In [ ]:
# 11.4 Historical Date Lookup & Ground Truth Benchmark Cell
print("=== HISTORICAL DATE LOOKUP WITH GROUND TRUTH ===")
date_query = input("Enter Timestamp (YYYY-MM-DD HH:MM) or type 'RANDOM' [Default: RANDOM]: ").strip()

if not date_query or date_query.upper() == 'RANDOM':
    test_start_idx = int(len(df_featured) * 0.9)
    sample_idx = np.random.randint(test_start_idx + 24, len(df_featured) - 25)
else:
    try:
        query_dt = pd.to_datetime(date_query)
        matches = df_raw.index[df_raw['time'] == query_dt].tolist()
        sample_idx = matches[0] if matches else (df_raw['time'] - query_dt).abs().idxmin()
        sample_idx = max(24, sample_idx)
    except Exception as e:
        print(f"Error parsing date: {e}. Using random test sample.")
        sample_idx = np.random.randint(int(len(df_featured) * 0.9) + 24, len(df_featured) - 25)

# Extract lookback window
hist_window = df_featured.iloc[sample_idx - 24:sample_idx]
actual_val = df_featured.iloc[sample_idx]['temperature_2m']
target_time = df_raw.iloc[sample_idx]['time']

t_s = target_scaler.transform(hist_window[['temperature_2m']])
f_s = feature_scaler.transform(hist_window[feature_cols])
tensor_in = torch.tensor(np.hstack([t_s, f_s]), dtype=torch.float32).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    pred_val = target_scaler.inverse_transform(model(tensor_in).cpu().numpy()).item()

error = abs(actual_val - pred_val)
acc = max(0, 100 - (error / max(1.0, actual_val) * 100))

print("-" * 65)
print(f" Target Timestamp  : {target_time}")
print(f" Ground Truth Temp : {actual_val:.2f} °C")
print(f" LSTM Prediction   : {pred_val:.2f} °C")
print(f" Absolute Error    : {error:.3f} °C")
print(f" Accuracy          : {acc:.2f} %")
print("-" * 65)


In [ ]:
# 11.5 Multi-Hour Forward Rollout Plotter
def plot_multi_hour_rollout(current_temp=28.0, humidity=60.0, pressure=915.0, hour=14, month=5, hours_ahead=12):
    _, df_w = predict_from_user_input(current_temp, humidity, pressure, hour=hour, month=month)
    
    # Auto-regressive rollout
    curr_df = df_w[all_feature_names].copy().reset_index(drop=True)
    rollout_preds = []
    
    for step in range(1, hours_ahead + 1):
        t_s = target_scaler.transform(curr_df[['temperature_2m']])
        f_s = feature_scaler.transform(curr_df[feature_cols])
        t_in = torch.tensor(np.hstack([t_s, f_s]), dtype=torch.float32).unsqueeze(0).to(device)
        
        with torch.no_grad():
            nxt_t = target_scaler.inverse_transform(model(t_in).cpu().numpy()).item()
            rollout_preds.append(nxt_t)
            
        new_row = curr_df.iloc[-1:].copy()
        new_row['temperature_2m'] = nxt_t
        curr_h = (int(new_row['hour'].values[0]) + 1) % 24
        new_row['hour'] = curr_h
        new_row['hour_sin'] = np.sin(2 * np.pi * curr_h / 24.0)
        new_row['hour_cos'] = np.cos(2 * np.pi * curr_h / 24.0)
        curr_df = pd.concat([curr_df.iloc[1:], new_row], ignore_index=True)

    # Plot
    plt.figure(figsize=(12, 5), facecolor='#0f172a')
    ax = plt.gca()
    ax.set_facecolor('#1e293b')
    
    past_h = list(range(-23, 1))
    future_h = list(range(1, hours_ahead + 1))
    
    plt.plot(past_h, df_w['temperature_2m'], color='#38bdf8', linewidth=2.2, label='Past 24h Telemetry (Input)', marker='o', markersize=4)
    plt.plot([0] + future_h, [current_temp] + rollout_preds, color='#fbbf24', linewidth=2.5, linestyle='--', label=f'LSTM {hours_ahead}-Hour Forecast', marker='s', markersize=5)
    
    plt.axvline(0, color='#ef4444', linestyle=':', label='Forecast Origin (T=0)')
    plt.title(f'Continuous Temperature Trajectory: 24h Lookback & {hours_ahead}h Future Forecast', fontsize=13, fontweight='bold', color='#f8fafc')
    plt.xlabel('Hours Relative to Now', color='#cbd5e1')
    plt.ylabel('Temperature (°C)', color='#cbd5e1')
    plt.tick_params(colors='#cbd5e1')
    plt.grid(True, linestyle='--', alpha=0.3, color='#334155')
    plt.legend(frameon=True, facecolor='#0f172a', edgecolor='#475569', labelcolor='#cbd5e1')
    plt.tight_layout()
    plt.show()

plot_multi_hour_rollout(current_temp=29.0, humidity=55.0, pressure=915.0, hour=14, month=5, hours_ahead=12)
